### Nikhil Agent Logging with Lifecycle Hooks

This notebook demonstrates using custom agent lifecycle hooks and MCP server management with new variable and file names.

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from nikhil_agent_logger import NikhilAgentLogger
from nikhil_mcp_manager import NikhilMCPManager
import os

In [ ]:
load_dotenv(override=True)

In [ ]:
sandbox_dir = os.path.abspath(os.path.join(os.getcwd(), 'sandbox'))
brave_key = os.getenv('BRAVE_API_KEY')

manager_configs = {
    'files': {'params': {'command': 'npx', 'args': ['-y', '@modelcontextprotocol/server-filesystem', sandbox_dir]}},
    'browser': {'params': {'command': 'npx', 'args': ['@playwright/mcp@latest']}},
    'brave': {'params': {'command': 'npx', 'args': ['-y', '@modelcontextprotocol/server-brave-search'], 'env': {'BRAVE_API_KEY': brave_key}}}
}

agent_instructions = """
You browse the internet to accomplish your instructions.
You are highly capable at browsing the internet independently to accomplish your task, 
including accepting all cookies and clicking 'not now' as appropriate to get to the content you need. If one website isn't fruitful, try another. 
Be persistent until you have solved your assignment, trying different options and sites as needed.
You search the web to find good information, and then use the web URLs to navigate to the content you need.
You can save files to the sandbox, and you can read files from the sandbox.
If you save the user's request to the sandbox, you say so, rather than replying with the file content or a summary of the file content.
"""

user_input = """
I want a good recipe for a 500g loaf of sourdough bread.
Use a search and find the best recipe from the results.
Then use your browser automation tools navigate to that recipe via its URL.
From the page full content, give a detailed summary of the recipe and save it to sandbox/recipe.md in markdown format.
If a page doesn't load, try another one.
"""

In [ ]:
async with NikhilMCPManager(manager_configs) as mcp_mgr:
    agent = Agent(
        name="NikhilResearcher",
        instructions=agent_instructions,
        model="gpt-4o-mini",
        mcp_servers=mcp_mgr.all_instances(),
        hooks=NikhilAgentLogger(label="NikhilResearcher")
    )
    with trace("investigate"):
        result = await Runner.run(agent, user_input)
        print("\n\n=== FINAL OUTPUT ===\n\n")
        print(result.final_output)